In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Precios PVPC reales 2023-2024
pvpc_prices = pd.read_csv("../data/pvpc_prices_2023_2024.csv")
pvpc_prices["datetime"] = pd.to_datetime(pvpc_prices["datetime"], utc=True)
pvpc_prices["datetime"] = pvpc_prices["datetime"].dt.tz_convert("Europe/Madrid")
pvpc_prices["Hour"] = pvpc_prices["datetime"].dt.hour

# Datos de CO2
co2_df = pd.read_csv("../data/co2_intensity_2023_2024.csv")
co2_df["datetime"] = pd.to_datetime(co2_df["datetime"], utc=True)
co2_df["datetime"] = co2_df["datetime"].dt.tz_convert("Europe/Madrid")
co2_df["Hour"] = co2_df["datetime"].dt.hour

# Centroides del clustering
centroids_df = pd.read_csv("../data/cluster_centroids.csv")

print("Precios PVPC:", pvpc_prices.shape)
print("Datos CO2:", co2_df.shape)
print("Centroides:", centroids_df.shape)

Precios PVPC: (17544, 4)
Datos CO2: (17544, 3)
Centroides: (4, 26)


In [2]:
# Merge por datetime para tener precio y CO2 en el mismo DataFrame
energy_df = pd.merge(
    pvpc_prices[["datetime", "Hour", "price_eur_kwh"]],
    co2_df[["datetime", "co2_g_kwh"]],
    on="datetime",
    how="inner"
)

print("Shape tras merge:", energy_df.shape)
print("\nPrimeras filas:")
display(energy_df.head())

print("\nComprobación de nulos:")
print(energy_df.isnull().sum())

Shape tras merge: (17544, 4)

Primeras filas:


,datetime,Hour,price_eur_kwh,co2_g_kwh
0,2023-01-01 00:00:00+01:00,0,0.04145,69.416094
1,2023-01-01 01:00:00+01:00,1,0.04301,78.661453
2,2023-01-01 02:00:00+01:00,2,0.05807,82.035862
3,2023-01-01 03:00:00+01:00,3,0.06069,83.748215
4,2023-01-01 04:00:00+01:00,4,0.06291,85.059920



Comprobación de nulos:
datetime         0
Hour             0
price_eur_kwh    0
co2_g_kwh        0
dtype: int64


Función de penalización por perfil (igual que v2)

In [3]:
cluster_labels = {
    0: "Tarde",
    1: "Bajo consumo",
    2: "Madrugador",
    3: "Alto consumo"
}

def get_profile_penalty(cluster_id, alpha=2.0):
    centroid_cols = [f"hour_{h}" for h in range(24)]
    centroid = centroids_df.loc[
        centroids_df["cluster"] == cluster_id, centroid_cols
    ].values[0]
    
    min_val = centroid.min()
    max_val = centroid.max()
    normalized = (centroid - min_val) / (max_val - min_val)
    penalty = 1 + alpha * normalized
    
    return penalty

Función recomendador v3 con precio + CO2 + perfil

In [4]:
def recommend_device_v3(
    energy_df,
    device_name,
    energy_kwh,
    date,
    cluster_id,
    allowed_start_hour=0,
    allowed_end_hour=23,
    alpha=2.0,
    beta=0.5,
    top_n=5
):
    """
    Recomienda las mejores horas combinando:
    - Precio PVPC real (€/kWh)
    - Penalización por perfil de consumo del hogar (alpha)
    - Penalización por emisiones de CO2 (beta)
    
    Coste ajustado = precio × penalización_perfil × penalización_co2 × energía
    """
    # Filtrar día
    day_data = energy_df[
        energy_df["datetime"].dt.date == pd.to_datetime(date).date()
    ].copy()

    day_data = day_data[
        (day_data["Hour"] >= allowed_start_hour) &
        (day_data["Hour"] <= allowed_end_hour)
    ].copy()

    # Penalización por perfil
    penalty_profile = get_profile_penalty(cluster_id, alpha=alpha)
    day_data["penalty_profile"] = day_data["Hour"].apply(
        lambda h: penalty_profile[h]
    )

    # Penalización por CO2 normalizada entre 1 y 1+beta
    co2_min = day_data["co2_g_kwh"].min()
    co2_max = day_data["co2_g_kwh"].max()
    if co2_max > co2_min:
        day_data["penalty_co2"] = 1 + beta * (
            (day_data["co2_g_kwh"] - co2_min) / (co2_max - co2_min)
        )
    else:
        day_data["penalty_co2"] = 1.0

    # Costes
    day_data["device_cost_eur"] = day_data["price_eur_kwh"] * energy_kwh
    day_data["co2_emissions_g"] = day_data["co2_g_kwh"] * energy_kwh
    day_data["adjusted_cost"] = (
        day_data["price_eur_kwh"] *
        day_data["penalty_profile"] *
        day_data["penalty_co2"] *
        energy_kwh
    )

    day_data["device"] = device_name

    recommendations = day_data.sort_values("adjusted_cost").head(top_n)

    return recommendations[[
        "device", "datetime", "Hour",
        "price_eur_kwh", "co2_g_kwh",
        "penalty_profile", "penalty_co2",
        "device_cost_eur", "co2_emissions_g",
        "adjusted_cost"
    ]]

Test del recomendador v3

In [5]:
date_test = "2023-06-15"

print("=== Perfil Madrugador — v3 con CO2 ===")
display(recommend_device_v3(
    energy_df=energy_df,
    device_name="Lavadora",
    energy_kwh=1.0,
    date=date_test,
    cluster_id=2,
    allowed_start_hour=7,
    allowed_end_hour=23,
    alpha=2.0,
    beta=0.5,
    top_n=5
))

=== Perfil Madrugador — v3 con CO2 ===


,device,datetime,Hour,price_eur_kwh,co2_g_kwh,penalty_profile,penalty_co2,device_cost_eur,co2_emissions_g,adjusted_cost
3975,Lavadora,2023-06-15 16:00:00+02:00,16,0.15613,95.506732,1.351299,1.023312,0.15613,95.506732,0.215897
3974,Lavadora,2023-06-15 15:00:00+02:00,15,0.15269,96.359116,1.378748,1.030735,0.15269,96.359116,0.216991
3973,Lavadora,2023-06-15 14:00:00+02:00,14,0.14935,93.922553,1.517549,1.009516,0.14935,93.922553,0.228803
3976,Lavadora,2023-06-15 17:00:00+02:00,17,0.15975,100.696557,1.471853,1.068509,0.15975,100.696557,0.251237
3972,Lavadora,2023-06-15 13:00:00+02:00,13,0.19522,92.829885,1.689139,1.000000,0.19522,92.829885,0.329754


Comparación v2 vs v3 para ver el impacto del CO2

In [6]:
from IPython.display import display

print("=== v2 sin CO2 — Perfil Madrugador ===")
# Simulamos v2 poniendo beta=0
display(recommend_device_v3(
    energy_df=energy_df,
    device_name="Lavadora",
    energy_kwh=1.0,
    date=date_test,
    cluster_id=2,
    allowed_start_hour=7,
    allowed_end_hour=23,
    alpha=2.0,
    beta=0.0,
    top_n=5
)[["device", "Hour", "price_eur_kwh", "co2_g_kwh", "device_cost_eur"]])

print("\n=== v3 con CO2 — Perfil Madrugador ===")
display(recommend_device_v3(
    energy_df=energy_df,
    device_name="Lavadora",
    energy_kwh=1.0,
    date=date_test,
    cluster_id=2,
    allowed_start_hour=7,
    allowed_end_hour=23,
    alpha=2.0,
    beta=0.5,
    top_n=5
)[["device", "Hour", "price_eur_kwh", "co2_g_kwh", "device_cost_eur"]])

=== v2 sin CO2 — Perfil Madrugador ===


,device,Hour,price_eur_kwh,co2_g_kwh,device_cost_eur
3974,Lavadora,15,0.15269,96.359116,0.15269
3975,Lavadora,16,0.15613,95.506732,0.15613
3973,Lavadora,14,0.14935,93.922553,0.14935
3976,Lavadora,17,0.15975,100.696557,0.15975
3982,Lavadora,23,0.18186,150.243266,0.18186



=== v3 con CO2 — Perfil Madrugador ===


,device,Hour,price_eur_kwh,co2_g_kwh,device_cost_eur
3975,Lavadora,16,0.15613,95.506732,0.15613
3974,Lavadora,15,0.15269,96.359116,0.15269
3973,Lavadora,14,0.14935,93.922553,0.14935
3976,Lavadora,17,0.15975,100.696557,0.15975
3972,Lavadora,13,0.19522,92.829885,0.19522


Función planificación v3 sin solape

In [8]:
def schedule_devices_v3(
    energy_df,
    devices,
    date,
    cluster_id,
    alpha=2.0,
    beta=0.5
):
    """
    Planifica varios dispositivos sin solape combinando
    precio PVPC + perfil de hogar + emisiones CO2.
    """
    scheduled_results = []
    occupied_hours = set()

    cluster_label = cluster_labels[cluster_id]

    for device in devices:
        day_data = energy_df[
            energy_df["datetime"].dt.date == pd.to_datetime(date).date()
        ].copy()

        day_data = day_data[
            (day_data["Hour"] >= device["allowed_start_hour"]) &
            (day_data["Hour"] <= device["allowed_end_hour"])
        ].copy()

        free_hours = day_data[
            ~day_data["Hour"].isin(occupied_hours)
        ].copy()

        if free_hours.empty:
            free_hours = day_data.copy()

        # Penalización perfil
        penalty_profile = get_profile_penalty(cluster_id, alpha=alpha)
        free_hours["penalty_profile"] = free_hours["Hour"].apply(
            lambda h: penalty_profile[h]
        )

        # Penalización CO2
        co2_min = free_hours["co2_g_kwh"].min()
        co2_max = free_hours["co2_g_kwh"].max()
        if co2_max > co2_min:
            free_hours["penalty_co2"] = 1 + beta * (
                (free_hours["co2_g_kwh"] - co2_min) / (co2_max - co2_min)
            )
        else:
            free_hours["penalty_co2"] = 1.0

        free_hours["device_cost_eur"] = (
            free_hours["price_eur_kwh"] * device["energy_kwh"]
        )
        free_hours["co2_emissions_g"] = (
            free_hours["co2_g_kwh"] * device["energy_kwh"]
        )
        free_hours["adjusted_cost"] = (
            free_hours["price_eur_kwh"] *
            free_hours["penalty_profile"] *
            free_hours["penalty_co2"] *
            device["energy_kwh"]
        )

        best = free_hours.sort_values("adjusted_cost").iloc[0]
        occupied_hours.add(int(best["Hour"]))

        scheduled_results.append({
            "device": device["device_name"],
            "cluster_label": cluster_label,
            "recommended_hour": int(best["Hour"]),
            "price_eur_kwh": round(best["price_eur_kwh"], 4),
            "co2_g_kwh": round(best["co2_g_kwh"], 2),
            "energy_kwh": device["energy_kwh"],
            "device_cost_eur": round(best["device_cost_eur"], 4),
            "co2_emissions_g": round(best["co2_emissions_g"], 2),
            "adjusted_cost": round(best["adjusted_cost"], 4)
        })

    return pd.DataFrame(scheduled_results)

Catálogo de dispositivos

In [9]:
devices_catalog = [
    {
        "device_name": "Lavadora",
        "energy_kwh": 1.0,
        "allowed_start_hour": 7,
        "allowed_end_hour": 23
    },
    {
        "device_name": "Lavavajillas",
        "energy_kwh": 1.3,
        "allowed_start_hour": 8,
        "allowed_end_hour": 23
    },
    {
        "device_name": "Secadora",
        "energy_kwh": 2.0,
        "allowed_start_hour": 9,
        "allowed_end_hour": 22
    },
    {
        "device_name": "Termo eléctrico",
        "energy_kwh": 2.5,
        "allowed_start_hour": 0,
        "allowed_end_hour": 23
    }
]

Test planificación v3

In [10]:
date_test = "2023-06-15"

print("=== Planificación v3 — Perfil Madrugador ===")
display(schedule_devices_v3(
    energy_df=energy_df,
    devices=devices_catalog,
    date=date_test,
    cluster_id=2,
    alpha=2.0,
    beta=0.5
))

print("\n=== Planificación v3 — Perfil Tarde ===")
display(schedule_devices_v3(
    energy_df=energy_df,
    devices=devices_catalog,
    date=date_test,
    cluster_id=0,
    alpha=2.0,
    beta=0.5
))

=== Planificación v3 — Perfil Madrugador ===


,device,cluster_label,recommended_hour,price_eur_kwh,co2_g_kwh,energy_kwh,device_cost_eur,co2_emissions_g,adjusted_cost
0,Lavadora,Madrugador,16,0.1561,95.51,1.0,0.1561,95.51,0.2159
1,Lavavajillas,Madrugador,15,0.1527,96.36,1.3,0.1985,125.27,0.2821
2,Secadora,Madrugador,14,0.1493,93.92,2.0,0.2987,187.85,0.4579
3,Termo eléctrico,Madrugador,3,0.1391,136.62,2.5,0.3477,341.55,0.4803



=== Planificación v3 — Perfil Tarde ===


,device,cluster_label,recommended_hour,price_eur_kwh,co2_g_kwh,energy_kwh,device_cost_eur,co2_emissions_g,adjusted_cost
0,Lavadora,Tarde,16,0.1561,95.51,1.0,0.1561,95.51,0.3606
1,Lavavajillas,Tarde,14,0.1493,93.92,1.3,0.1942,122.10,0.4832
2,Secadora,Tarde,15,0.1527,96.36,2.0,0.3054,192.72,0.7452
3,Termo eléctrico,Tarde,5,0.1361,138.21,2.5,0.3402,345.52,0.4774


Función baseline v3

In [11]:
def baseline_device_costs_v3(
    energy_df,
    devices,
    date,
    cluster_id,
    baseline_hour=20
):
    day_data = energy_df[
        energy_df["datetime"].dt.date == pd.to_datetime(date).date()
    ].copy()

    baseline_row = day_data[day_data["Hour"] == baseline_hour].iloc[0]
    baseline_price = baseline_row["price_eur_kwh"]
    baseline_co2 = baseline_row["co2_g_kwh"]
    cluster_label = cluster_labels[cluster_id]

    results = []
    for device in devices:
        results.append({
            "device": device["device_name"],
            "cluster_label": cluster_label,
            "baseline_hour": baseline_hour,
            "baseline_price_eur_kwh": round(baseline_price, 4),
            "baseline_co2_g_kwh": round(baseline_co2, 2),
            "energy_kwh": device["energy_kwh"],
            "baseline_cost_eur": round(baseline_price * device["energy_kwh"], 4),
            "baseline_co2_g": round(baseline_co2 * device["energy_kwh"], 2)
        })

    return pd.DataFrame(results)

Función comparación v3

In [12]:
def compare_v3_vs_baseline(
    energy_df,
    devices,
    date,
    cluster_id,
    alpha=2.0,
    beta=0.5,
    baseline_hour=20
):
    schedule = schedule_devices_v3(
        energy_df=energy_df,
        devices=devices,
        date=date,
        cluster_id=cluster_id,
        alpha=alpha,
        beta=beta
    )

    baseline = baseline_device_costs_v3(
        energy_df=energy_df,
        devices=devices,
        date=date,
        cluster_id=cluster_id,
        baseline_hour=baseline_hour
    )

    comparison = pd.merge(
        baseline,
        schedule,
        on=["device", "cluster_label", "energy_kwh"]
    )

    comparison["savings_eur"] = (
        comparison["baseline_cost_eur"] - comparison["device_cost_eur"]
    )
    comparison["savings_pct"] = (
        comparison["savings_eur"] / comparison["baseline_cost_eur"]
    ) * 100

    comparison["co2_savings_g"] = (
        comparison["baseline_co2_g"] - comparison["co2_emissions_g"]
    )
    comparison["co2_savings_pct"] = (
        comparison["co2_savings_g"] / comparison["baseline_co2_g"]
    ) * 100

    return comparison

Test comparación v3